In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
from math import ceil

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from scipy.stats.kde import gaussian_kde
from constants import PJM_SITE_NAMES

In [ ]:
PJM_DATA_DIR = Path("../../data/pjm")
PJM_DATA_FREQUENCY = "1h"

## Exploratory Data Analysis

### Global

In [ ]:
global_data_file_name = "pjm_hourly_est.csv"
global_data_file_path = PJM_DATA_DIR / global_data_file_name

global_df = pl.read_csv(
    global_data_file_path,
    columns=["Datetime"] + PJM_SITE_NAMES,
    schema_overrides={**{"Datetime": pl.Datetime}, **{site: pl.Float32 for site in PJM_SITE_NAMES}},
    new_columns=["timestamp"],
)

global_df = global_df.unpivot(index="timestamp", value_name="demand_mw", variable_name="site")
min_max_timestamps_by_site = (
    global_df
    .filter(pl.col("demand_mw").is_not_null())
    .group_by(pl.col("site"))
    .agg(
        min_timestamp=pl.col("timestamp").min(),
        max_timestamp=pl.col("timestamp").max()
    )
    .sort(by="min_timestamp")
)

fig, ax = plt.subplots()
ax.scatter(
    min_max_timestamps_by_site["min_timestamp"].to_numpy(),
    min_max_timestamps_by_site["site"].to_list(),
    alpha=0.75,
    label="min_timestamp",
)
ax.scatter(
    min_max_timestamps_by_site["max_timestamp"].to_numpy(),
    min_max_timestamps_by_site["site"].to_list(),
    alpha=0.75,
    label="max_timestamp",
)
ax.legend()
ax.grid(axis="y", alpha=0.3, ls="--")
ax.set(title="Min/Max Timestamps by Site");

### Local

In [ ]:
site = "PJMW"
data_file_name = f"{site}_hourly.csv"
data_file_path = PJM_DATA_DIR / data_file_name
site_df = pl.read_csv(
    data_file_path,
    columns=["Datetime", f"{site}_MW"],
    schema={"Datetime": pl.Datetime, f"{site}_MW": pl.Float64},
    new_columns=["timestamp"]
)
site_df = site_df.sort(by="timestamp")

In [ ]:
# Are there any duplicate timestamps?

def has_duplicate_timestamps(df: pl.DataFrame) -> bool:
    return df["timestamp"].is_duplicated().any()


def deduplicate_timestamps(df: pl.DataFrame) -> pl.DataFrame:
    """
    Deduplicates demand data by averaging observations for multiple timestamps
    """
    return (
        df
        .group_by("timestamp")
        .agg(pl.col(f"{site}_MW").mean())
        .sort(by="timestamp")
    )

if has_duplicate_timestamps(site_df):
    site_df = deduplicate_timestamps(site_df)
    assert not has_duplicate_timestamps(site_df)

In [ ]:
# Do timestamps match hourly frequency?

expected_timestamps = pl.datetime_range(
    start=site_df["timestamp"].min(),
    end=site_df["timestamp"].max(),
    interval=PJM_DATA_FREQUENCY,
    closed="both",
    eager=True,
)

expected_timestamps_df = (
    expected_timestamps
    .to_frame("expected_timestamp")
    .join(site_df, left_on="expected_timestamp", right_on="timestamp", how="full")
)

missing_timestamps_df = expected_timestamps_df.filter(pl.col("timestamp").is_null())
plt.scatter(
    missing_timestamps_df["expected_timestamp"].to_list(),
    np.ones(shape=(len(missing_timestamps_df, ))),
    s=20,
    color="grey",
    marker="x",
)
plt.title(f"Number of missing timestamps: {len(missing_timestamps_df)}");

In [ ]:
# Interpolate missing timestamps
def interpolate_site_data(pjm_df: pl.DataFrame) -> pl.DataFrame:
    expected_timestamps = pl.datetime_range(
        start=pjm_df["timestamp"].min(),
        end=pjm_df["timestamp"].max(),
        interval=PJM_DATA_FREQUENCY,
        closed="both",
        eager=True,
    )
    pjm_df = (
        expected_timestamps
        .to_frame("expected_timestamp")
        .join(pjm_df, left_on="expected_timestamp", right_on="timestamp", how="left")
        .sort(by=("expected_timestamp"))
        .select(pl.col("expected_timestamp"), pl.col(f"{site}_MW").interpolate())
        .rename(mapping={"expected_timestamp": "timestamp"})
    )
    return pjm_df

site_df = interpolate_site_data(pjm_df=site_df)

### Monthly Seasonality

In [ ]:
# Plot rolling demand timeseries for each year

min_year = site_df["timestamp"].min().year
max_year = site_df["timestamp"].max().year
years = list(range(min_year, max_year + 1))

# Normalise everything to same year
norm_year = 2000
assert norm_year <= min_year

rolling_demand_df = (
    site_df
    .rolling(index_column="timestamp", period="1d")
    .agg(rolling_mean=pl.col(f"{site}_MW").mean())
)

cmap = plt.cm.managua
vmin, vmax = -0.1, 0.9

fig, ax = plt.subplots(1, 1, figsize=(15, 3.5))
for i, year in enumerate(years):
    rolling_year_df = rolling_demand_df.filter(pl.col("timestamp").dt.year() == year).sort(by="timestamp")
    timestamps = [
        timestamp.replace(year=norm_year)
        for timestamp in rolling_year_df["timestamp"].to_list()
    ]
    color_val = vmin + (vmax - vmin) * i / len(years)
    ax.plot(
        timestamps,
        rolling_year_df["rolling_mean"].to_list(),
        color=cmap(color_val),
        label=f"{year}",
    )

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_minor_locator(mdates.MonthLocator())
ax.grid(which="major", ls="--", color="grey", alpha=0.5)
ax.set(title=f"Rolling Daily Demand For {site}", ylabel="Demand (MW)");

fig.tight_layout();

In [ ]:
min_year = site_df["timestamp"].min().year
max_year = site_df["timestamp"].max().year
years = list(range(min_year, max_year + 1))

# Normalise everything to same year
norm_year = 2000
assert norm_year <= min_year

monthly_demand_df = (
    site_df
    .group_by(pl.col("timestamp").dt.truncate("1mo"))
    .agg(mean_demand=pl.col(f"{site}_MW").mean())
    .sort(by="timestamp")
)

cmap = plt.cm.managua
vmin, vmax = -0.1, 0.9

fig, ax = plt.subplots(1, 1, figsize=(15, 3.5))
for i, year in enumerate(years):
    year_df = monthly_demand_df.filter(pl.col("timestamp").dt.year() == year).sort(by="timestamp")
    timestamps = [
        timestamp.replace(year=norm_year)
        for timestamp in year_df["timestamp"].to_list()
    ]
    color_val = vmin + (vmax - vmin) * i / len(years)
    ax.plot(
        timestamps,
        year_df["mean_demand"].to_list(),
        color=cmap(color_val),
        label=f"{year}",
        lw=2.5,
    )

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_minor_locator(mdates.MonthLocator())
ax.grid(which="major", ls="--", color="grey", alpha=0.5)
ax.set(title=f"Average Monthly Demand For {site}", ylabel="Demand (MW)");

# Only show every third year?

fig.tight_layout();

### Hourly & Weekday Seasonality

In [ ]:
# Day / weekday seasonality: Scatter plot of demand vs hour colored by weekday

weekdays = list(range(1, 8))
months = list(range(1, 13))

cmap = plt.cm.managua
vmin, vmax = -0.1, 0.9

n_rows, n_cols = 3, 4
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3), sharex=True, sharey=True)
axes = axes.flatten()

for i, month in enumerate(months):
    month_df = site_df.filter(pl.col("timestamp").dt.month() == month)
    for weekday in weekdays:
        weekday_df = (
            month_df
            .filter(pl.col("timestamp").dt.weekday() == weekday)
            .with_columns(hour=pl.col("timestamp").dt.hour())
            .sort(by="timestamp")
        )
        mean_hourly_demand = (
            weekday_df
            .group_by("hour")
            .agg(demand_mean=pl.col(f"{site}_MW").mean())
            .sort(by="hour")
        )
    
        color_val = vmin + (vmax - vmin) * weekday / len(weekdays)
        axes[i].scatter(
            weekday_df["hour"].to_list(),
            weekday_df[f"{site}_MW"].to_list(),
            color=cmap(color_val),
            alpha=0.35,
            s=10,
        )
        axes[i].plot(
            mean_hourly_demand["hour"].to_list(),
            mean_hourly_demand["demand_mean"].to_list(),
            color=cmap(color_val),
            lw=2.5,
            label=f"Weekday={weekday}",
        )
    
    axes[i].grid(which="major", ls="--", lw=0.5, alpha=0.5, color="grey")
    axes[i].set(title=f"Month = {month}")
    if i % n_cols == 0:
        axes[i].set(ylabel="Demand (MW)")
    if i // n_cols == n_rows - 1:
        axes[i].set(xlabel="Hour of day")
    

# Title and legend for whole plot

fig.align_labels();
fig.tight_layout();

In [ ]:
# FFT spectrum

fig, ax = plt.subplots()

demand_data = site_df.sort(by="timestamp")[f"{site}_MW"].to_numpy()
demand_data_scaled = demand_data - demand_data.mean()
demand_fft = np.fft.rfft(demand_data_scaled)
freq_fft = np.fft.rfftfreq(n=len(demand_data_scaled), d=1)
mask = freq_fft < 0.2
ax.plot(
    freq_fft[mask],
    np.abs(demand_fft[mask]),
    color=plt.cm.berlin(0.5),
)

ax.axvline(0, color="black", ls="-", lw=1.5)
ax.axhline(0, color="black", ls="-", lw=1.5)

half_daily_freq = 1 / 12
ax.axvline(half_daily_freq, color="tab:red", ls="--", lw=1.5, label="Half-Daily")

daily_freq = 1 / 24
ax.axvline(daily_freq, color="tab:orange", ls="--", lw=1.5, label="Daily")

weekly_freq = 1 / (24 * 7)
ax.axvline(weekly_freq, color="tab:blue", ls="--", lw=1.5, label="Weekly")

ax.legend()

### Distribution

In [ ]:
min_year = site_df["timestamp"].min().year
max_year = site_df["timestamp"].max().year
years = list(range(min_year, max_year + 1))
months = list(range(1, 13))

cmap = plt.cm.managua
vmin, vmax = -0.1, 0.9

n_rows, n_cols = 3, 4
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3), sharex=True, sharey=True)
axes = axes.flatten()

for i, month in enumerate(months):
    month_df = site_df.filter(pl.col("timestamp").dt.month() == month)
    for year_idx, year in enumerate(years):
        year_month_df = month_df.filter(pl.col("timestamp").dt.year() == year)
        if len(year_month_df) < 2: continue
        kde = gaussian_kde(year_month_df[f"{site}_MW"].to_list())
        x = np.linspace(
            year_month_df[f"{site}_MW"].min() * 0.8,
            year_month_df[f"{site}_MW"].max() * 1.2,
            1000
        )
        
        color_val = vmin + (vmax - vmin) * year_idx / len(years)
        axes[i].plot(x, kde(x), color=cmap(color_val), lw=1.5)
    
    axes[i].grid(which="major", ls="--", lw=0.5, alpha=0.5, color="grey")
    axes[i].set(title=f"Month={month}")
    if i % n_cols == 0:
        axes[i].set(ylabel="Density")
    if i // n_cols == n_rows - 1:
        axes[i].set(xlabel="Demand (MW)")

fig.align_labels()
fig.tight_layout();

In [ ]:
### How do mean / std vary over time?

### Autocorrelation